In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio
from matplotlib.patches import Rectangle

PROJECT_DIR = Path.cwd().parent
SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from core.config import P, settings
from core.setup import setup_logging, setup_project_path

setup_logging(settings.LOG_LEVEL)
setup_project_path(PROJECT_DIR)
# setup_data()

CLEANED_DIR = settings.data_dir / "cleaned"
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

In [ ]:
from core.config import Parameters

BoxSource = pd.DataFrame | Path | None


def _boxes_in_window(source: BoxSource, start: float, window_s: float, score_threshold: float | None) -> pd.DataFrame:
    if source is None:
        return pd.DataFrame()
    df = source if isinstance(source, pd.DataFrame) else pd.read_csv(source, sep="\t")
    if score_threshold is not None and "Score" in df.columns:
        df = df[df["Score"] >= score_threshold]
    return df[(df["End Time (s)"] > start) & (df["Begin Time (s)"] < start + window_s)]


def _draw_boxes(ax, rows: pd.DataFrame, start: float, color: str) -> None:
    for _, row in rows.iterrows():
        x0, y0 = row["Begin Time (s)"] - start, row["Low Freq (Hz)"]
        width, height = row["End Time (s)"] - row["Begin Time (s)"], row["High Freq (Hz)"] - y0
        ax.add_patch(Rectangle((x0, y0), width, height, edgecolor=color, facecolor="none", linewidth=1.5))
        label = f"{row['Species']}/{row['Call type']}"
        if "Score" in row:
            label += f" {row['Score']:.2f}"
        ax.text(x0, y0 + height, label, color=color, fontsize=10, fontweight="bold", va="bottom")


def plot_clips_with_boxes(
    wav_path: Path,
    annotations: BoxSource = None,
    detections: BoxSource = None,
    window_s: float = 10.0,
    score_threshold: float | None = None,
    params: Parameters = P,
) -> None:
    sources = {"GT": (annotations, "cyan"), "modelo": (detections, "lime")}
    native_sr = sf.info(wav_path).samplerate
    duration_s = sf.info(wav_path).duration
    freqs_hz = np.linspace(0.0, params.target_sr / 2, params.n_fft // 2 + 1)
    spectrogram = torchaudio.transforms.Spectrogram(
        n_fft=params.n_fft, win_length=params.win_length, hop_length=params.hop_length, power=2.0
    )

    for start in np.arange(0.0, duration_s, window_s):
        waveform, _ = torchaudio.load(
            wav_path, frame_offset=int(start * native_sr), num_frames=int(window_s * native_sr)
        )
        waveform = waveform.mean(dim=0)
        if native_sr != params.target_sr:
            waveform = torchaudio.functional.resample(waveform, native_sr, params.target_sr)
        spec_db = 10 * torch.log10(spectrogram(waveform) + params.eps)

        _, ax = plt.subplots(figsize=(10, 4))
        ax.imshow(spec_db.numpy(), origin="lower", aspect="auto",
                  extent=(0, window_s, freqs_hz[0], freqs_hz[-1]), cmap="magma")

        handles = []
        for label, (source, color) in sources.items():
            rows = _boxes_in_window(source, start, window_s, score_threshold)
            if rows.empty:
                continue
            _draw_boxes(ax, rows, start, color)
            handles.append(Rectangle((0, 0), 1, 1, edgecolor=color, facecolor="none", label=label))
        if handles:
            ax.legend(handles=handles, loc="lower right", fontsize=9)

        ax.set_xlabel("Tiempo (s)")
        ax.set_ylabel("Frecuencia (Hz)")
        ax.set_title(f"{Path(wav_path).name} @ {start:.1f}s")
        plt.show()

In [ ]:
from infer import load_model
from pipelines.inference_pipeline import predict

device = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint_path = PROJECT_DIR / "checkpoints" / "128d_64q_iouiou_10cls_best.pth"
model, labels = load_model(checkpoint_path, device)

wav_path = CLEANED_DIR / "weddells_saddleBack_tamarin__LW/240125_0028.wav"
ann_path = wav_path.with_suffix(".txt")

detections = predict(model, wav_path, labels, device, score_threshold=0.8)
plot_clips_with_boxes(wav_path, annotations=ann_path, detections=detections, score_threshold=0.8)